# K-TIDE — LoRA SFT on Colab (T4) with W&B

Trains one condition, logs it live to the W&B project `K-TIDE`, merges and packages the model for
the evaluation notebook, and generates answers for the **during_development** split under the
official conditions (CUDA FP16, greedy). Held-out test answers are produced only for the final
comparison on the frozen eval set.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. Colab secrets (key icon, notebook access on): `WANDB_API_KEY` (W&B), `HF_TOKEN` (Hugging Face,
   write access, for uploading results).
3. Build the bundle locally (`python -m ktide.colab_bundle ...`), upload it to the dataset repo under
   `bundles/`, and paste the printed `BUNDLE_SHA256` below.

`RUN_MODE = "smoke"` trains a few optimizer steps and stops (W&B check, tag `smoke`, kept out of the
course/research reports). `RUN_MODE = "full"` runs everything. After a runtime restart, run from
the first code cell again. W&B receives metrics and settings only: no data, answers, or model.

In [ ]:
# ==== 0. settings ====
HF_REPO = "HunBot/K-TIDE-Dataset"         # dataset repo with bundles/ and results/
BUNDLE = "ktide_bundle_m1_data-v1.zip"    # file name under bundles/
BUNDLE_SHA256 = "cb79079441547d9fd73a00b208bff8f0e7b2dbe1ebb6364ed6d0e37f0014662f"  # from ktide.colab_bundle
QA = ["data_private/train/m1_lecture_only/qa.jsonl"]
VARIANT = "translated"                    # translated (M1, M3) | preserved (M2, M4)
CONDITION = "M1"
DATA = "data-v1"                          # training data label (used in names)
TRACK = "research"                        # research: fixed common recipe | course: assignment tuning
RUN_MODE = "smoke"                        # smoke (a few steps, W&B check) | full
SMOKE_STEPS = 8
EPOCHS, LR, R, ALPHA, TARGETS = 3, 1e-4, 16, 32, "all-linear"
PRECISION = "fp16"                        # fixed for every research run (matches FP16 evaluation)
UPLOAD_MODEL_ZIP = False                  # True only for the final assignment model (about 3 GB)

In [ ]:
# ==== 1. setup: packages, secrets, bundle (hash-checked), code path ====
import glob, hashlib, json, os, shutil, subprocess, sys, zipfile
assert BUNDLE_SHA256, "paste the BUNDLE_SHA256 printed by ktide.colab_bundle"
assert RUN_MODE in ("smoke", "full") and TRACK in ("research", "course")

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"command failed: {' '.join(cmd[:6])} ...")

# same versions as the evaluation notebook, plus peft (LoRA) and wandb (torch: Colab preinstalled)
sh([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "transformers==4.57.1", "accelerate==1.11.0", "huggingface_hub==0.36.0",
    "safetensors==0.6.2", "peft==0.17.1", "pydantic>=2.9", "wandb==0.31.2"])

from google.colab import userdata
for key in ("WANDB_API_KEY", "HF_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
        print(key, "loaded")
    except Exception as e:
        print(key, "not available:", type(e).__name__)
os.environ.update({
    "WANDB_PROJECT": "K-TIDE", "WANDB_ENTITY": "hun-bot",
    "WANDB_LOG_MODEL": "false", "WANDB_WATCH": "false", "WANDB_DISABLE_CODE": "true",
    "TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
})

from huggingface_hub import HfApi, hf_hub_download
revision = HfApi().dataset_info(HF_REPO).sha          # pin the dataset revision for this run
bundle = hf_hub_download(HF_REPO, f"bundles/{BUNDLE}", repo_type="dataset", revision=revision)
digest = hashlib.sha256(open(bundle, "rb").read()).hexdigest()
assert digest == BUNDLE_SHA256, f"bundle hash {digest} != BUNDLE_SHA256 (stale or wrong bundle)"

WORK = f"/content/ktide/{digest[:12]}"                 # one folder per bundle version
if not os.path.isfile(f"{WORK}/bundle_manifest.json"):
    shutil.rmtree(WORK, ignore_errors=True)
    zipfile.ZipFile(bundle).extractall(WORK)
manifest = json.load(open(f"{WORK}/bundle_manifest.json"))
bad = [p for p, h in manifest["files"].items()
       if hashlib.sha256(open(f"{WORK}/{p}", "rb").read()).hexdigest() != h]
assert not bad, f"files differ from the manifest: {bad[:5]}"
assert manifest["split_check"] is None or manifest["split_check"]["passed"], "split check failed"
os.chdir(WORK)
os.environ["PYTHONPATH"] = f"{WORK}/src"
os.environ["KTIDE_HF_DATASET_REVISION"] = revision
sys.path.insert(0, f"{WORK}/src")

def run(*args):
    """Run `python -m <module> ...` in WORK, stream its output, return the output lines."""
    p = subprocess.Popen([sys.executable, "-u", "-m", *args], cwd=WORK, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"{args[0]} failed (exit {p.returncode}); see the output above")
    return lines

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> T4 GPU"
print("python", sys.version.split()[0], "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))
print("bundle", manifest["tag"], "| git", (manifest["git_commit"] or "")[:7],
      "dirty" if manifest["git_dirty"] else "", "| dataset revision", revision[:7])
for f in QA:
    assert f in manifest["files"], f"training file not in bundle: {f}"

In [ ]:
# ==== 2. token-level loss-masking check on the real data ====
run("ktide.train.data", "--qa", *QA, "--variant", VARIANT, "--check")

In [ ]:
# ==== 3. LoRA SFT (live W&B run; the output folder is named by the W&B run id) ====
import secrets, string
RUN_ID = "".join(secrets.choice(string.ascii_lowercase + string.digits) for _ in range(8))
smoke = RUN_MODE == "smoke"
args = ["ktide.train.sft", "--qa", *QA, "--variant", VARIANT, "--condition", CONDITION,
        "--out-root", "/content/outputs", "--run-id", RUN_ID, "--report-to", "wandb",
        "--track", "smoke" if smoke else TRACK, "--data-label", DATA,
        "--bundle-manifest", f"{WORK}/bundle_manifest.json",
        "--epochs", str(EPOCHS), "--lr", str(LR), "--r", str(R), "--alpha", str(ALPHA),
        "--targets", TARGETS, "--precision", PRECISION]
if smoke:
    args += ["--max-steps", str(SMOKE_STEPS)]
out_line = [l for l in run(*args) if l.startswith("OUTPUT_DIR=")]
OUT = out_line[-1].split("=", 1)[1].strip()
print("run:", f"https://wandb.ai/hun-bot/K-TIDE/runs/{RUN_ID}", "| output:", OUT)
if smoke:
    print("smoke test done: check the W&B run, then set RUN_MODE = 'full' and run again")

In [ ]:
# ==== 4. merge + package + verify (professor unpack_model, greedy decoding, size limit) ====
if RUN_MODE == "full":
    run("ktide.train.export", "--adapter", f"{OUT}/adapter", "--out", f"{OUT}/package")

In [ ]:
# ==== 5. answers for the during_development split (professor load_model + answer_questions) ====
if RUN_MODE == "full":
    run("ktide.generate", "--experiment", "main", "--condition", CONDITION,
        "--model-dir", f"{OUT}/package/qwen25_sft_model",
        "--questions-file", manifest["dev_questions"],
        "--tag", f"colab-t4-{DATA}-{RUN_ID}",
        "--notes", f"{CONDITION} LoRA, variant={VARIANT}, {DATA}, wandb={RUN_ID}")

In [ ]:
# ==== 6. pack results (named by the run id) and upload them to the Hugging Face repo ====
if RUN_MODE == "full":
    run_dir = glob.glob(f"{WORK}/runs/main/{CONDITION}/*{RUN_ID}*")[-1]
    res = f"/content/results/{RUN_ID}"
    shutil.rmtree(res, ignore_errors=True)
    shutil.copytree(run_dir, f"{res}/run/{os.path.basename(run_dir)}")
    shutil.copytree(f"{OUT}/adapter", f"{res}/adapter")
    for f in ("train_config.json", "train_log.json"):
        shutil.copy(f"{OUT}/{f}", res)
    shutil.copy(f"{OUT}/package/export_report.json", res)
    archive = shutil.make_archive(res, "zip", res)
    target = f"results/{CONDITION}/{DATA}/{RUN_ID}.zip"
    print("results zip:", archive, os.path.getsize(archive) // 1024, "KB")
    try:
        api = HfApi()
        api.upload_file(path_or_fileobj=archive, path_in_repo=target, repo_id=HF_REPO,
                        repo_type="dataset", commit_message=f"{CONDITION} {DATA} {RUN_ID}")
        if UPLOAD_MODEL_ZIP:
            api.upload_file(path_or_fileobj=f"{OUT}/package/qwen25_sft_model.zip",
                            path_in_repo=f"models/{CONDITION}/{DATA}/{RUN_ID}.zip",
                            repo_id=HF_REPO, repo_type="dataset")
        print("uploaded:", target, "\nlocal: uv run python -m ktide.import_colab --hf", target)
    except Exception as e:
        print("upload failed (results kept at", archive, "):", type(e).__name__, e)

In [ ]:
# (optional) download the results zip directly, e.g. when the upload failed (full mode only)
if RUN_MODE == "full" and "archive" in globals():
    from google.colab import files
    files.download(archive)
else:
    print("nothing to download: results are packed only in RUN_MODE = 'full'")
